# 03 · Lift-Splat-Shoot → triplane (TPV), step by step

**`TRIPLANE_LIFTERS["lift_splat"]` → `lifting.triplane.LiftSplatTriplaneLifter`**  ·  Philion & Fidler, *Lift, Splat, Shoot* (ECCV 2020)

> Lift every pixel to D weighted 3D points (depth probability × context), splat (scatter-sum) the point cloud into voxels, and shoot by collapsing the volume along Z, Y and X.

This notebook shows, cell by cell, how the **Lift-Splat-Shoot** lifting technique produces a
**tri-perspective view** (three orthogonal feature planes `xy` = BEV, `xz`, `yz`) instead of a single
bird's-eye-view map, *without changing a line of the existing BEV lifter* - the triplane module wraps it
(composition) and only adds what is needed to keep the height axis.

| § | what you will see |
|---|---|
| 1 | the idea, the animated step-by-step GIF and a detailed Mermaid diagram |
| 2 | every animation step as a still image |
| 3 | the shared toy scene (4 cameras, vehicles, pedestrians) |
| 4 | building the triplane lifter from the registry |
| 5 | **inside the lifter**: every intermediate tensor, with exact checks |
| 6 | the triplane contract, verified |
| 7 | end-to-end training on 3D semantic occupancy |
| 8 | the negative control: does it really use the cameras? |
| 9 | how to use it in your own code, recap, cheat sheet, exercises |

## 1. The idea

![Lift-Splat-Shoot triplane lifting animation](../../assets/triplane/lift_splat.gif)


Lift-Splat-Shoot is the **push** counterpart of Depth-Warp:

1. **Lift** - every pixel `(u, v)` and depth bin `d` becomes a 3D point `Cameras.unproject(u, v, d)` carrying the
   feature `p(d | u, v) · c(u, v)`. That is `D · h · w` points per camera.
2. **Splat** - every point is added to the voxel it falls in (`voxel_pooling`, scatter-sum). Mass is conserved: the
   sum of the voxel volume equals the sum of all in-grid point features (checked below).
3. **Shoot** - originally: a BEV CNN. Here: `VoxelToTriplane` collapses the splatted volume into `xy`, `xz` and `yz`.

The depth head, the frustum points and the splat are `LiftSplatLifter`'s, unchanged.

### Detailed diagram

Colours: <span style="color:#d48806">■ data</span> · <span style="color:#1f77b4">■ fixed geometry</span>
· <span style="color:#8e44ad">■ learned</span> · <span style="color:#2e8b57">■ reused from the existing BEV lifter</span>
· <span style="color:#c0392b">■ outputs</span>. (Mermaid renders in JupyterLab ≥ 4.1, VS Code and on GitHub;
the matplotlib version below renders everywhere.)

```mermaid
flowchart LR
  IMG["N camera images<br/>(B, N, 3, H, W)"]:::data --> ENC["ImageEncoder<br/>pyramid (B, N, C, H_l, W_l)"]:::learned
  CAM["Cameras<br/>K (B,N,3,3) · cam_to_ego (B,N,4,4)"]:::data
  subgraph CORE["LiftSplatLifter (existing, unchanged)"]
    direction TB
    DH["DepthHead<br/>p(d | pixel) · context c"]:::learned --> LIFT["Lift: outer product<br/>p(d) ⊗ c → D·H·W weighted points"]:::reused
    FP["frustum_points<br/>Cameras.unproject(pixel, d_bin)"]:::geo --> SPLAT
    LIFT --> SPLAT["Splat: voxel_pooling (scatter-sum)<br/>points PUSHED into voxels"]:::geo
  end
  ENC --> DH
  CAM --> FP
  SPLAT --> VOX["voxel volume (B, C', X, Y, Z)"]:::reused
  subgraph V2T["VoxelToTriplane (new)"]
    direction TB
    CXY["fold Z into channels<br/>HeightCompressor → xy"]:::learned
    CXZ["fold Y into channels<br/>HeightCompressor → xz"]:::learned
    CYZ["fold X into channels<br/>HeightCompressor → yz"]:::learned
  end
  VOX --> CXY & CXZ & CYZ
  CXY & CXZ & CYZ --> TPV
  TPV["TPVPlanes<br/>xy (B,C,X,Y) · xz (B,C,X,Z) · yz (B,C,Y,Z)"]:::out
  TPV --> BEVH["TriplaneBEVHead<br/>f = xy + xz + yz → fold Z<br/>BEV (B, C, X, Y)"]:::learned
  TPV --> AGG["TPVAggregator<br/>voxel logits (B,K,X,Y,Z)<br/>+ point logits (B,K,P)"]:::learned
  classDef data fill:#fff7e6,stroke:#d48806,color:#222
  classDef geo fill:#eaf2fb,stroke:#1f77b4,color:#222
  classDef learned fill:#f1e6f8,stroke:#8e44ad,color:#222
  classDef reused fill:#eaf7ea,stroke:#2e8b57,color:#222
  classDef out fill:#fdecea,stroke:#c0392b,color:#222
```

### Shapes and equations


$$ P_{n,d,u,v} = \mathrm{unproject}_n(u, v, d), \qquad f_{n,d,u,v} = p_\theta(d \mid u, v)\, c(u, v) $$

$$ V[i, j, k] = \sum_{n,d,u,v} \mathbb{1}\big[P_{n,d,u,v} \in \mathrm{voxel}(i, j, k)\big]\ f_{n,d,u,v} $$

| tensor | shape |
|---|---|
| depth / context | `(B, N, D, h, w)` / `(B, N, C', h, w)` |
| frustum points | `(B, N, D·h·w, 3)` metric |
| voxel volume | `(B, C', X, Y, Z)` (sum-pooled) |

In [1]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

# tpv_kit lives next to this notebook; also works when Jupyter starts at the repo root.
for _p in (".", os.path.join("notebooks", "triplane")):
    if os.path.isdir(os.path.join(_p, "tpv_kit")):
        sys.path.insert(0, os.path.abspath(_p)); break

import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt

from lifting import BaseLifter
from lifting.encoders import SimpleConvEncoder
from lifting.lifters import LIFTERS
from lifting.triplane import TRIPLANE_LIFTERS, build_triplane_lifter, build_triplane_model, build_triplane_bev_model
from tpv_kit import (TriplaneScene, PlanePlotter, PipelineDiagram, StoryboardViewer,
                     TriplaneContractChecker, OccupancyExperiment, ExperimentResult)

torch.manual_seed(0)
torch.set_num_threads(min(4, torch.get_num_threads()))
plt.rcParams.update({"figure.dpi": 100, "image.interpolation": "nearest"})
plotter = PlanePlotter()
print("torch", torch.__version__, "| triplane lifters:", TRIPLANE_LIFTERS.names())

torch 2.14.1 | triplane lifters: ['alpamayo', 'bev_adapter', 'bevformer', 'depth_warp', 'lift_splat', 'mlp_view', 'simple_bev', 'tiim', 'tpvformer']


In [2]:
PipelineDiagram('Lift-Splat-Shoot → triplane').extend([
    ('DepthHead', 'p(d) · context', 'learned'),
    ('Lift', 'D·h·w weighted\n3D points / camera', 'reused'),
    ('Splat', 'voxel_pooling\nscatter-sum (push)', 'reused'),
    ('voxel volume', "(B, C', X, Y, Z)", 'reused'),
    ('Shoot', 'VoxelToTriplane\nfold Z / Y / X', 'learned'),
    ('TPVPlanes', 'xy · xz · yz', 'learned'),
]).show()

## 2. The animation, one still per step

The GIF above is rendered by `lifting.triplane.animation` (Pillow only, exact geometry on a toy scene:
4 cameras at the origin, a car, a pedestrian and a pole). `StoryboardViewer` re-renders the **final frame of every
step** so the whole sequence can be read at once. Regenerate all GIFs with `python scripts/make_triplane_gifs.py`.

In [3]:
viewer = StoryboardViewer("lift_splat")
for i, title in enumerate(viewer.step_titles(), 1):
    print(f"step {i}: {title}")
viewer.contact_sheet()

step 1: Inputs
step 2: Per-pixel depth distribution
step 3: Lift: pixel -> weighted 3D points
step 4: Splat: scatter-sum into voxels
step 5: Shoot: collapse Z, Y and X
step 6: Triplane output


## 3. The scene

Every triplane notebook uses the same `TriplaneScene`: a `32 m × 32 m × 4 m` region cut into `16 × 16 × 4`
cells, four `48 × 96` cameras whose yaw is **randomly rotated per scene** (so no network can memorise where
"front" is - it must read the calibration), and procedurally placed vehicles (red) and pedestrians (blue).
The ground truth is a 3D occupancy grid `(B, X, Y, Z)`; collapsing it along each axis gives the three
**label planes** each triplane is supervised against.

In [4]:
scene = TriplaneScene()
print(scene.describe())
batch = scene.sample
plotter.cameras(batch.images, title="the four camera images of test scene 0")
plotter.labels([("ground truth", scene.label_planes(batch.occupancy))], title="what the three planes should encode")

grid      : x (-16.0, 16.0) y (-16.0, 16.0) z (-1.0, 3.0) m, resolution (16, 16, 4)
voxel     : (2.0, 2.0, 1.0) m
cameras   : 4 x (48, 96) px
scenes    : 128 train / 32 test (batch 4)
TPV cells : xy 256 + xz 64 + yz 64 = 384  (voxels: 1024)


## 4. Build the triplane lifter

Two equivalent ways:

* `TRIPLANE_LIFTERS.build("lift_splat", grid=..., in_channels=..., out_channels=...)` - the raw registry;
* `build_triplane_lifter("lift_splat", grid, channels, image_size, num_cameras, encoder)` - also injects the
  context some lifters need (`feature_size`, `num_cameras`, `depth_bins`), exactly like `lifting.models.build_lifter`
  does for BEV lifters.

The image encoder turns `(B·N, 3, H, W)` images into a feature pyramid (strides 4 and 8 here); every lifter
consumes `(B, N, C, H_l, W_l)` per level.

In [5]:
C = 32
encoder = SimpleConvEncoder(out_channels=C, num_levels=2)
features = scene.encode(encoder, batch)
cameras = batch.cameras
print("feature pyramid:", [tuple(f.shape) for f in features])

lifter = build_triplane_lifter("lift_splat", scene.grid, C, scene.image_size, scene.num_cameras, encoder)
print(type(lifter).__name__, "| is a BaseLifter too:", isinstance(lifter, BaseLifter))
print(f"parameters: {sum(p.numel() for p in lifter.parameters()):,}")
for name, child in lifter.named_children():
    n = sum(p.numel() for p in child.parameters())
    print(f"  .{name:<12s} {type(child).__name__:<28s} {n:>9,} params")

feature pyramid: [(4, 4, 32, 12, 24), (4, 4, 32, 6, 12)]
LiftSplatTriplaneLifter | is a BaseLifter too: True
parameters: 384,248
  .bev_head     TriplaneBEVHead                 37,984 params
  .core         LiftSplatLifter                 11,128 params
  .to_planes    VoxelToTriplane                335,136 params


## 5. Inside the lifter, step by step

### 5.1 Lift: depth distribution × context, placed at frustum points

In [6]:
depth_p, context = lifter.depth_and_context(features)         # (B,N,D,h,w), (B,N,C',h,w)
h, w = context.shape[-2:]
cams_hw = cameras.scaled_to(h, w)                              # intrinsics rescaled to the feature map
points = lifter.core.frustum_points(cams_hw, h, w)             # (B, N, D*h*w, 3)
print("depth", tuple(depth_p.shape), "| context", tuple(context.shape), "| frustum points", tuple(points.shape))
inside = scene.grid.contains(points)
print(f"points per camera: {points.shape[2]:,}   inside the grid: {inside.float().mean():.1%}")

depth (4, 4, 24, 12, 24) | context (4, 4, 32, 12, 24) | frustum points (4, 4, 6912, 3)
points per camera: 6,912   inside the grid: 30.7%


In [7]:
# where do the frustum points of camera 0 land? (top view, coloured by depth bin)
pts0 = points[0, 0].view(lifter.depth_bins.num_bins, h, w, 3)[:, h // 2]   # middle image row
fig, ax = plt.subplots(figsize=(5, 5), facecolor="white")
for d in range(pts0.shape[0]):
    ax.scatter(pts0[d, :, 1], pts0[d, :, 0], s=4, color=plt.cm.viridis(d / pts0.shape[0]))
g = scene.grid
ax.add_patch(plt.Rectangle((g.y_bounds[0], g.x_bounds[0]), g.extents[1], g.extents[0], fill=False, ec="k", lw=1.2))
ax.invert_xaxis(); ax.set_xlabel("y (m, left +)"); ax.set_ylabel("x (m, forward +)"); ax.set_aspect("equal")
ax.set_title("camera 0, middle row: one point per (column, depth bin)", fontsize=10)
plt.tight_layout(); plt.show()

### 5.2 Splat: scatter-sum into voxels (mass conservation check)

`voxel_pooling(..., reduce="sum")` adds every in-grid point feature to its voxel. Summing the whole volume therefore
equals summing the features of all points that fell inside the grid - an exact identity checked below.

In [8]:
voxels = lifter.lift_voxels(features, cameras)               # (B, C', X, Y, Z)
lifted = (depth_p[:, :, :, None] * context[:, :, None])        # (B, N, D, C', h, w)
lifted = lifted.permute(0, 1, 2, 4, 5, 3).reshape(points.shape[0], -1, context.shape[2])
_, in_grid = scene.grid.flat_voxel_index(points.reshape(points.shape[0], -1, 3))
expected = (lifted * in_grid[..., None]).sum(1)                 # (B, C')
print("voxel volume:", tuple(voxels.shape))
print("sum over voxels == sum of in-grid point features:",
      torch.allclose(voxels.sum((2, 3, 4)), expected, rtol=1e-4, atol=1e-3))

voxel volume: (4, 32, 16, 16, 4)
sum over voxels == sum of in-grid point features: True


In [9]:
# how many frustum points land in each voxel? (dense near the cameras, sparse far away)
idx, ok = scene.grid.flat_voxel_index(points[0].reshape(-1, 3))
counts = torch.bincount(idx[ok], minlength=int(np.prod(scene.grid.resolution))).float().view(*scene.grid.resolution)
plotter.scalar_planes([counts.sum(2), counts.sum(1), counts.sum(0)],
                      "frustum points per cell (summed along the collapsed axis)", cmap="magma")

### Collapse the volume into three planes: `VoxelToTriplane`

A BEV lifter ends with a `HeightCompressor`: fold `Z` into the channels (`C·Z`) and convolve over `(X, Y)`.
`VoxelToTriplane` does that **three times**, each time moving a different axis to the end:

| plane | permute `(B, C, X, Y, Z)` → | folded axis | conv over |
|---|---|---|---|
| `xy` | `(B, C, X, Y, Z)` | `Z` | `(X, Y)` |
| `xz` | `(B, C, X, Z, Y)` | `Y` | `(X, Z)` |
| `yz` | `(B, C, Y, Z, X)` | `X` | `(Y, Z)` |

**Why fold rather than average?** Folding keeps *where along the collapsed axis* a feature was: the `xz` plane
can still tell a car on the left from one on the right. The next cell proves it on a toy volume.

In [10]:
from lifting.triplane import VoxelToTriplane
from lifting.geometry import GridSpec

tiny = GridSpec(resolution=(2, 3, 2))
to_planes = VoxelToTriplane(in_channels=1, grid=tiny, out_channels=4).eval()
a = torch.zeros(1, 1, 2, 3, 2); a[..., 0, 0, 1] = 1.0      # feature at y = 0 (left)
b = torch.zeros(1, 1, 2, 3, 2); b[..., 0, 2, 1] = 1.0      # same feature at y = 2 (right)
with torch.no_grad():
    pa, pb = to_planes(a), to_planes(b)
print("mean over Y identical?      ", torch.equal(a.mean(3), b.mean(3)), "  <- averaging would lose left vs right")
print("folded xz planes identical? ", torch.allclose(pa.xz, pb.xz), " <- VoxelToTriplane keeps it")
print("compressor weights (C_in·folded axis -> C_out):",
      {k: tuple(m.weight.shape) for k, m in to_planes.named_modules() if isinstance(m, torch.nn.Conv2d)})

mean over Y identical?       True   <- averaging would lose left vs right
folded xz planes identical?  False  <- VoxelToTriplane keeps it
compressor weights (C_in·folded axis -> C_out): {'compress_xy.conv.0.0': (4, 2, 3, 3), 'compress_xy.conv.1': (4, 4, 1, 1), 'compress_xz.conv.0.0': (4, 3, 3, 3), 'compress_xz.conv.1': (4, 4, 1, 1), 'compress_yz.conv.0.0': (4, 2, 3, 3), 'compress_yz.conv.1': (4, 4, 1, 1)}


In [11]:
voxels = lifter.lift_voxels(features, cameras)
planes = lifter.to_planes(voxels)
print("voxels", tuple(voxels.shape), "->", {k: tuple(getattr(planes, k).shape) for k in ("xy", "xz", "yz")})
print(f"memory: voxels {voxels[0].numel():,} floats per scene vs planes "
      f"{sum(getattr(planes, k)[0].numel() for k in ('xy', 'xz', 'yz')):,}")

voxels (4, 32, 16, 16, 4) -> {'xy': (4, 32, 16, 16), 'xz': (4, 32, 16, 4), 'yz': (4, 32, 16, 4)}
memory: voxels 32,768 floats per scene vs planes 12,288


## 6. The triplane contract, verified

Every triplane lifter promises the same five properties (Liskov: it is still a `BaseLifter`, so it drops into
`BEVSegmentationModel`, the trainer and the verifier unchanged). `TriplaneContractChecker` runs them on real tensors.

In [12]:
assert TriplaneContractChecker(lifter, features, cameras).run()
with torch.no_grad():
    planes = lifter.lift_planes(features, cameras)
plotter.features(planes, title="untrained planes (joint PCA → RGB): structure comes only from geometry + random weights")

PASS  plane shapes            xy(4, 32, 16, 16) xz(4, 32, 16, 4) yz(4, 32, 16, 4)
PASS  BEV = bev_head(planes)  BEV (4, 32, 16, 16), max |diff| = 0.0e+00
PASS  f = xy + xz + yz        voxel (8,8,2) max |diff| = 0.0e+00
PASS  reuses BEV lifter       core = LiftSplatLifter = LIFTERS['lift_splat']
PASS  gradients reach images  1/2 pyramid levels receive gradient


## 7. Train end to end on 3D semantic occupancy

`build_triplane_model("lift_splat", ...)` = encoder → **this triplane lifter** → `TPVAggregator` (an MLP on
`f(x,y,z) = xy + xz + yz`), plus one auxiliary classifier per plane. `OccupancyTask(aux_weight=0.5)` adds a
cross-entropy on each plane against the collapsed labels of §3. 150 AdamW steps on 128 scenes, CPU only.

In [13]:
exp = OccupancyExperiment(scene, "lift_splat")
result = exp.run(steps=150)
print(ExperimentResult.header()); print(result.row())
OccupancyExperiment.plot_losses([result], title="Lift-Splat-Shoot triplane: occupancy training loss")

lifter          params    time   before    after  corrupted   verdict
lift_splat        521k   27.1s    0.004    0.388      0.028   uses geometry


In [14]:
out = exp.predict()
pred = out["voxel_logits"].argmax(1)                       # (B, X, Y, Z) class ids
plotter.labels([("ground truth", scene.label_planes(batch.occupancy)),
                ("prediction", scene.label_planes(pred))],
               title="held-out scene: occupancy collapsed onto the three planes")
plotter.features(out["planes"], title="trained feature planes (joint PCA → RGB)")

### Training with an explicit depth loss

`Lift-Splat` above learned its depth distribution **only through the occupancy loss**. Here the same model, data,
seed and 150 steps are trained with `lifting.training.DepthSupervisedTask`, which wraps `OccupancyTask` and adds a
cross-entropy between every `DepthHead`'s bins and the rendered depth (BEVDepth, Li et al. AAAI 2023,
[arXiv:2206.10092](https://arxiv.org/abs/2206.10092)). It hooks the existing `DepthHead`, so the lifter is unchanged.

Diagnostics on the held-out batch: mean **entropy** of the per-pixel distribution (uniform over 24 bins =
ln 24 ≈ 3.18) and the correlation between **expected depth** `E[d] = Σ p(d)·d` and the true nearest-surface depth.

**Weight.** The depth term is weighted `0.05`. A sweep over `{0, 0.05, 0.1, 0.3, 1.0}` x `{150, 400}` steps on this
scene showed: every non-zero weight teaches depth (corr > 0.9), but weights ≥ 0.1 cost occupancy mIoU here because
depth then dominates the 150-step budget; at `0.05` mIoU stays within noise of the occupancy-only run.


In [15]:
from lifting.training import DepthSupervisedTask, OccupancyTask
from lifting.training.depth_supervision import DEPTH_PREDICTIONS

exp_depth = OccupancyExperiment(scene, "lift_splat", depth_weight=0.05)  # same seed / data / steps
result_depth = exp_depth.run(steps=150)
result_depth.name = "+depth loss"
print(ExperimentResult.header()); print(result.row()); print(result_depth.row())


def depth_diagnostics(experiment):
    """Entropy and corr(E[d], true depth) of the trained DepthHead on the held-out batch."""
    probe = DepthSupervisedTask(OccupancyTask(3))          # only used to capture the DepthHead output
    with torch.no_grad():
        (probs, bins), = probe.forward(experiment.model.eval(), batch)[DEPTH_PREDICTIONS]
    return DepthSupervisedTask.entropy_and_correlation(probs, batch.depth.flatten(0, 1), bins)


print()
for label, experiment in (("occupancy only", exp), ("+ depth loss", exp_depth)):
    entropy, corr = depth_diagnostics(experiment)
    print(f"{label:15s} mean entropy {entropy:.3f} (uniform = {np.log(24):.3f}) | corr(E[d], true depth) = {corr:+.3f}")


lifter          params    time   before    after  corrupted   verdict
lift_splat        521k   27.1s    0.004    0.388      0.028   uses geometry
+depth loss       521k   21.9s    0.004    0.383      0.027   uses geometry

occupancy only  mean entropy 3.082 (uniform = 3.178) | corr(E[d], true depth) = +0.343
+ depth loss    mean entropy 2.183 (uniform = 3.178) | corr(E[d], true depth) = +0.944


## 8. Negative control: does it really use the cameras?

Same images, same weights - but every camera's extrinsic is rotated by 90° about the vertical axis. A lifter that reads
pixels **through** the calibration now puts every object in the wrong place and its mIoU collapses; a geometry-free
lifter would not notice. (This is the triplane version of the repo's `lifting-verify` check.)

In [16]:
corrupted = exp.predict(corrupted=True)["voxel_logits"].argmax(1)
plotter.labels([("ground truth", scene.label_planes(batch.occupancy)),
                ("correct calib.", scene.label_planes(pred)),
                ("cams yawed 90°", scene.label_planes(corrupted))],
               title="same weights, wrong extrinsics")
print(f"mIoU  correct calibration: {result.miou_after:.3f}   |   cameras yawed 90°: {result.miou_corrupted:.3f}"
      f"   ->  {'uses geometry ✅' if result.uses_geometry else 'does NOT use geometry'}")

mIoU  correct calibration: 0.388   |   cameras yawed 90°: 0.028   ->  uses geometry ✅


## 9. Use it in your own code

```python
from lifting.triplane import TRIPLANE_LIFTERS, build_triplane_model, build_triplane_bev_model

# 1) just the lifter: features (B, N, C, H_l, W_l) per level + Cameras
lifter = TRIPLANE_LIFTERS.build("lift_splat", grid=grid, in_channels=64, out_channels=64)
planes = lifter.lift_planes(features, cameras)   # TPVPlanes(xy, xz, yz)
bev    = lifter(features, cameras)               # (B, 64, X, Y) - drop-in BEV lifter

# 2) 3D semantic occupancy + point queries
model = build_triplane_model("lift_splat", grid, num_classes=3, aux_heads=True)
out = model(images, cameras, points=lidar_xyz)   # planes, voxel_logits, point_logits, aux_logits

# 3) BEV segmentation read off the triplane (same API as lifting.build_bev_model)
bev_model = build_triplane_bev_model("lift_splat", grid, num_classes=3)
```

In [17]:
bev_model = build_triplane_bev_model("lift_splat", scene.grid, 3, scene.image_size, scene.num_cameras, channels=16)
with torch.no_grad():
    logits = bev_model(batch.images, batch.cameras)["logits"]
print("BEV segmentation logits read off the triplane:", tuple(logits.shape))

BEV segmentation logits read off the triplane: (4, 3, 16, 16)


## Recap


* **Lift** = outer product of a depth distribution and a context feature at frustum points; **Splat** = scatter-sum.
* Push (Lift-Splat) vs pull (Depth-Warp): same frustum, opposite direction - pushing conserves feature mass, pulling
  gives every voxel exactly one read per camera.
* The triplane version keeps the lift and splat untouched and "shoots" into three planes instead of one BEV map.
* Trained on occupancy alone the depth head stays near-uniform; `DepthSupervisedTask` (BEVDepth-style depth cross-entropy) teaches it at no occupancy cost with a small weight (§7).


**Cheat sheet**

| concept | file |
|---|---|
| lift + splat (unchanged) | `src/lifting/lifters/lift_splat_lifter.py` |
| scatter-sum op | `src/lifting/ops/voxel_pooling.py` |
| triplane lifter | `src/lifting/triplane/lift_splat_triplane.py` |
| axis collapses | `src/lifting/triplane/plane_compressor.py` |
| triplane contract | `src/lifting/triplane/base.py` |
| BEV read-off from planes | `src/lifting/triplane/triplane_bev_head.py` |
| occupancy model / factories | `src/lifting/triplane/triplane_occupancy_model.py`, `factory.py` |
| this animation | `src/lifting/triplane/animation/storyboards/lift_splat_storyboard.py` |

**Exercises**

1. Use `reduce="mean"` instead of sum in a subclass. Which plane changes most, and why (hint: point density)?
2. Halve the number of depth bins. How do the frustum-point counts and the mIoU change?
3. Compare the learned planes with Depth-Warp's (notebook 02) on the same scene.

Next: see `00_triplane_overview.ipynb` for all techniques side by side.